# Broadcast Pipeline — Google Colab Educational Guide

This notebook is a **complete walkthrough** of the broadcast video processing pipeline:
from raw `.mp4` input to per-camera on-screen text timelines (scoreboard graphics, logos, etc.).

## Table of contents

1. [Learning outcomes](#learning-outcomes)
2. [Pipeline architecture](#pipeline-architecture)
3. [Prerequisites](#prerequisites)
4. [Runtime profiles (free vs Pro)](#runtime-profiles)
5. [Environment setup](#environment-setup)
6. [Configuration](#configuration)
7. [Stage-by-stage deep dive](#stage-by-stage-deep-dive)
8. [Chunked execution](#chunked-execution)
9. [Human-in-the-loop reference editing](#human-in-the-loop)
10. [Full end-to-end run](#full-end-to-end-run)
11. [Results exploration](#results-exploration)
12. [Optional timeline web UI](#optional-timeline-web-ui)
13. [Standalone OCR debug](#standalone-ocr-debug)
14. [Troubleshooting](#troubleshooting)
15. [Out of scope](#out-of-scope)

## Learning outcomes

After working through this notebook you will understand:

1. **The problem** — turn a sports broadcast into structured per-camera text timelines.
2. **The workflow** — ten ordered stages orchestrated by `run_pipeline()` in `src/broadcast_pipeline/orchestrator.py`.
3. **The artifacts** — every CSV/JSON/JPEG written to disk, and how to resume or partially re-run.
4. **Colab constraints** — memory, session timeouts, GPU vs CPU trade-offs, and Drive persistence.

The CLI entry point `main.py` is equivalent to calling `run_pipeline(PipelineConfig(...))` programmatically.
**Do not** run `!python main.py` on Colab without fixing `sys.path` — `main.py` only adds `src/`, but inner modules import `from src.camera_assignemnt...` and `from src.person_appearance...` which require the **repo root** on the path (see setup below).


## Pipeline architecture

```mermaid
flowchart TD
    video[Input video MP4] --> meta[meta: probe FPS/duration]
    meta --> extract[extract: scene cuts + frame sampling]
    extract --> filter[filter: full_court vs closeup]
    filter --> appearance[appearance: person colors]
    appearance --> cameras[cameras: cluster camera_id]
    cameras --> ocr[ocr: RapidOCR + readability]
    ocr --> reference[reference: discover approved text]
    reference --> enrich[enrich: temporal IoU stabilization]
    enrich --> associate[associate: map OCR to reference]
    associate --> aggregate[aggregate: per-camera timelines]
    aggregate --> outputs[CSVs + pipeline_summary.json]
```

**Stage order** (`STAGE_ORDER` in `src/broadcast_pipeline/config.py`):

`meta → extract → filter → appearance → cameras → ocr → reference → enrich → associate → aggregate`

### Expected outputs

| Artifact | Stage | Purpose |
|----------|-------|---------|
| `video_meta.json` | meta | FPS, duration, resolution, frame count |
| `scenes.json` | extract | Scene cut boundaries (frame + time) |
| `frame_index.csv` | extract | Sampled frames with `camera` / `ocr` roles |
| `frames/*.jpg` | extract | Extracted frame JPEGs |
| `scene_types.csv` | filter | `full_court` vs `closeup` per scene (feeds appearance constraints) |
| `frame_appearance.csv` | appearance | Per-frame person count + clothing colors |
| `scene_appearance.csv` | appearance | Scene-level appearance signature for camera constraints |
| `scene_assignments.csv` | cameras | Camera ID per scene (majority vote) |
| `frame_assignments.csv` | cameras | Camera ID per sampled frame |
| `frame_ocr.csv` | ocr | Raw OCR words + detections per frame |
| `approved_text_reference.csv` | reference | User-editable approved text list |
| `frame_ocr_enriched.csv` | enrich | Temporally stabilized OCR |
| `frame_text_associated.csv` | associate | OCR mapped to reference text |
| `dropped_text.csv` | associate | Tokens that failed association |
| `aggregated_complete.csv` | aggregate | Per-camera complete-text timeline |
| `aggregated_partial.csv` | aggregate | Per-camera partial-text timeline |
| `pipeline_summary.json` | aggregate | Run statistics |


## Prerequisites

### Hardware / Colab runtime

| Requirement | Free tier | Colab Pro GPU |
|-------------|-----------|---------------|
| Runtime | CPU OK for text stages; GPU helps cameras | T4/A100 recommended for ensemble cameras |
| RAM | ≥12 GB; use free profile (`hsv` cameras) | ≥16 GB for full ResNet50+DINOv2 ensemble |
| Disk | Mount Google Drive; outputs include many JPEGs | Same |
| Session length | Use chunked cells + `resume=True` | Can often run end-to-end |

### Software

- **Python:** `pyproject.toml` declares `>=3.13`. Colab may ship 3.10–3.12. Install packages directly via pip in the setup cell below — **avoid** `pip install -e .` if it rejects your Python version.
- **System packages:** None required beyond pip wheels (OpenCV, PySceneDetect, etc.).
- **Network:** First run downloads torchvision weights (Pro profile), DINOv2, RapidOCR ONNX models, and YOLO11n-seg ONNX for appearance (~hundreds of MB total, cached per session).

### Import path (critical)

Inner modules use `from src.camera_assignemnt...`, `from src.scene_ocr...`, and `from src.person_appearance...`. You must add **both** the repo root and `src/` to `sys.path`:

```python
ROOT = Path("/content/CV_problem")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
```

### Project acquisition (upload zip)

1. Zip the project locally. **Exclude** `data/`, `graphify-out/`, `.git/` to keep size small.
2. Upload the zip to Colab Files or Google Drive.
3. Unzip via the setup cell (checks for `src/broadcast_pipeline/`, not just an empty folder).
4. Set `ROOT` and `ZIP_PATH` in the configuration / unzip cells to match your Drive layout.

#### Zip manifest

| Path | Required | Purpose |
|------|----------|---------|
| `src/broadcast_pipeline/` | Yes | Pipeline orchestration |
| `src/scene_ocr/` | Yes | OCR stage |
| `src/person_appearance/` | Yes | Person segmentation + clothing colors (`appearance` stage) |
| `src/camera_assignemnt/` | Yes | Camera clustering + scene filter |
| `static/timeline_viz/` | Yes (for web UI) | Timeline visualization assets |
| `main.py` | Optional | CLI reference |
| `models/scene_mlp.joblib` | Optional | Better scene filter; HSV fallback if missing |
| `data/`, `graphify-out/`, `.git/` | Exclude | Not needed on Colab |

### Video input

- Supply your own `.mp4` broadcast clip (no sample video ships with the repo).
- **Start with 1–3 minutes** for your first run.
- Upload to Drive or Colab and set `VIDEO_PATH` below.

### Disk estimate

Approximate sampled frames per scene:

`camera_samples_per_scene + scene_duration_sec × ocr_samples_per_sec`

Multiply by number of scenes (depends on cut frequency) to estimate JPEG count and Drive usage.

### Optional dependency extras

| Extra | Packages | Needed for |
|-------|----------|------------|
| embedding | torch, torchvision | Camera ensemble (Pro profile) |
| ocr | `rapidocr>=3.8` + onnxruntime | OCR stage (`use_det` / `text_det` / `text_rec` API) |
| appearance | onnxruntime | YOLO11n-seg ONNX person segmentation (`appearance` stage) |
| appearance-export | ultralytics + onnx | One-time export of `yolo11n-seg.pt` → ONNX (setup cell below) |
| ocr-vlm | openai | `enable_vlm=True` |
| viz | fastapi, uvicorn, python-multipart | Timeline web UI |

### Secrets

- `OPENAI_API_KEY` — only if `enable_vlm=True`. Set via Colab **Secrets** (`userdata`) or `os.environ`.


## Runtime profiles

Choose `RUNTIME_PROFILE` before running setup. The free profile avoids torch entirely by setting `ensemble_method="hsv"`.

> **Preflight quirk:** `preflight()` gates torch on `ensemble_method == "ensemble"`, **not** on `fast_cameras`. Setting `fast_cameras=True` alone still requires torch unless you also set `ensemble_method="hsv"`.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# --- User configuration (edit these paths) ---
from pathlib import Path

RUNTIME_PROFILE = "pro"  # "free" | "pro"

PROFILES = {
    "free": dict(
        fast_cameras=True,
        ensemble_method="hsv",  # required to skip torch in preflight
        ocr_scale=1.25,
        ocr_samples_per_sec=1.0,
        camera_samples_per_scene=3,
        enable_vlm=False,
    ),
    "pro": dict(
        fast_cameras=False,
        ensemble_method="ensemble",
        ocr_scale=1.5,
        ocr_samples_per_sec=2.0,
        camera_samples_per_scene=5,
        enable_vlm=False,
    ),
}

# After unzipping the project zip:
ROOT = Path("/content/drive/MyDrive/Delete/SportsBiz")

# Video and output — use Drive paths for persistence across disconnects:
VIDEO_PATH = Path("/content/drive/MyDrive/Delete/SportsBiz/data/Untitled_sample.mp4")
OUTPUT_DIR = Path("/content/drive/MyDrive/Delete/SportsBiz/data/pipeline")

print(f"Profile: {RUNTIME_PROFILE}")
print(f"ROOT: {ROOT}")
print(f"Video: {VIDEO_PATH}")
print(f"Output: {OUTPUT_DIR}")


Profile: pro
ROOT: /content/drive/MyDrive/Delete/SportsBiz
Video: /content/drive/MyDrive/Delete/SportsBiz/data/Untitled_sample.mp4
Output: /content/drive/MyDrive/Delete/SportsBiz/data/pipeline


## Environment setup

Run these cells once per Colab session.

In [3]:
# Optional: check GPU (Pro runtime)
import shutil
if shutil.which("nvidia-smi"):
    !nvidia-smi
else:
    print("No GPU detected — free profile (hsv cameras) is appropriate.")


Wed Jun 17 19:07:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Mount Google Drive (recommended for persistence)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab — skip Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Unzip project — run after uploading zip to Drive or /content/
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/Delete/SportsBiz/code_optimized_f3.zip")  # edit to your zip path on Drive
FORCE_REEXTRACT = False  # set True to unzip again even if ROOT folder exists


def _project_ready(path: Path) -> bool:
    return (path / "src" / "broadcast_pipeline").is_dir()


def _resolve_project_root(base: Path) -> Path | None:
    """Find directory containing src/broadcast_pipeline (handles nested zip folders)."""
    if _project_ready(base):
        return base
    if not base.is_dir():
        return None
    for child in sorted(base.iterdir()):
        if child.is_dir() and _project_ready(child):
            return child
    return None


resolved = _resolve_project_root(ROOT)
if resolved is not None and not FORCE_REEXTRACT:
    if resolved != ROOT:
        print(f"Project code at {resolved} (nested inside {ROOT})")
    else:
        print(f"Project code found at {ROOT}")
    ROOT = resolved
elif ZIP_PATH.is_file():
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        print(f"Extracting {ZIP_PATH.name} ({len(zf.namelist())} entries) into {ROOT}")
        zf.extractall(ROOT)
    resolved = _resolve_project_root(ROOT)
    if resolved is None:
        raise FileNotFoundError(
            f"Zip extracted to {ROOT} but src/broadcast_pipeline not found. "
            "Re-zip with src/ at the top level, or set ROOT to the nested folder."
        )
    if resolved != ROOT:
        print(f"Project code at {resolved} (update ROOT in config cell if needed)")
    ROOT = resolved
    print(f"Ready: {ROOT / 'src' / 'broadcast_pipeline'}")
else:
    raise FileNotFoundError(
        f"No project at {ROOT} and zip not found at {ZIP_PATH}. "
        "Upload zip, set ZIP_PATH, or copy src/ into ROOT."
    )


Extracting code_optimized_f3.zip (858 entries) into /content/drive/MyDrive/Delete/SportsBiz
Ready: /content/drive/MyDrive/Delete/SportsBiz/src/broadcast_pipeline


In [8]:
import sys

# Re-resolve in case kernel was restarted after unzip cell
if "_resolve_project_root" in globals():
    _found = _resolve_project_root(ROOT)
    if _found is not None:
        ROOT = _found

_marker = ROOT / "src" / "broadcast_pipeline"
assert _marker.is_dir(), (
    f"Missing {_marker}. Run the unzip cell above or set ROOT to the folder containing src/."
)
for entry in (str(ROOT), str(ROOT / "src")):
    if entry not in sys.path:
        sys.path.insert(0, entry)
print(f"ROOT={ROOT}")
print("sys.path (head):", sys.path[:4])


ROOT=/content/drive/MyDrive/Delete/SportsBiz
sys.path (head): ['/content/drive/MyDrive/Delete/SportsBiz/src', '/content/drive/MyDrive/Delete/SportsBiz', '/content', '/env/python']


In [9]:
# Install dependencies (direct pip — avoids pyproject python>=3.13 constraint)
%pip install -q opencv-python scenedetect pandas scikit-learn scipy hdbscan matplotlib numpy joblib

if RUNTIME_PROFILE == "pro":
    %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
    %pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null || true
    %pip install -q onnxruntime-gpu

%pip install -q "rapidocr>=3.8" onnxruntime

# Appearance stage: export YOLO11n-seg to ONNX once per environment
%pip install -q "ultralytics>=8.3" onnx
import subprocess
subprocess.run(
    ["python", str(ROOT / "scripts" / "download_person_seg_model.py")],
    check=False,
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 65.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 102.6 MB/s eta 0:00:00


CompletedProcess(args=['python', '/content/drive/MyDrive/Delete/SportsBiz/scripts/download_person_seg_model.py'], returncode=0)

In [10]:
# Verify imports
import importlib
import importlib.metadata

modules = [
    "broadcast_pipeline",
    "broadcast_pipeline.orchestrator",
    "src.scene_ocr.extractor",
    "src.person_appearance.segmenter",
    "src.camera_assignemnt.embedding_cluster.pipeline",
]
for name in modules:
    importlib.import_module(name)
    print(f"OK  {name}")

import pandas as pd
import cv2
rapidocr_ver = importlib.metadata.version("rapidocr")
print(f"pandas {pd.__version__}, opencv {cv2.__version__}, rapidocr {rapidocr_ver}")

from rapidocr import RapidOCR

_probe = RapidOCR()
required = ("use_det", "text_det", "text_rec", "use_cls")
missing = [name for name in required if not hasattr(_probe, name)]
if missing:
    raise RuntimeError(
        f"rapidocr {rapidocr_ver} is incompatible. "
        f"Missing RapidOCR attributes: {missing}. Install rapidocr>=3.8."
    )
print("OK  RapidOCR API (text_det / text_rec present)")
del _probe

seg_model = ROOT / "models" / "person_seg" / "yolo11n-seg.onnx"
if not seg_model.is_file():
    print(f"WARN  person-seg ONNX missing at {seg_model} — re-run setup cell")
else:
    print(f"OK  person-seg ONNX ({seg_model.stat().st_size // 1024} KB)")

from src.accelerator.install import check_gpu_stack, gpu_install_hints

gpu_report = check_gpu_stack("auto")
print(gpu_report)
if gpu_report.ocr_cuda_missing or gpu_report.ocr_coreml_missing or gpu_report.torch_cuda_missing:
    print("GPU install hints:")
    print("\n".join(gpu_install_hints(gpu_report)))


OK  broadcast_pipeline
OK  broadcast_pipeline.orchestrator
OK  src.scene_ocr.extractor
OK  src.person_appearance.segmenter
OK  src.camera_assignemnt.embedding_cluster.pipeline
pandas 2.2.2, opencv 4.13.0, rapidocr 3.8.4


[INFO] 2026-06-17 19:10:08,109 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 19:10:08,143 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-17 19:10:08,144 [RapidOCR] main.py:65: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-17 19:10:08,330 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 19:10:08,334 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-17 19:10:08,335 [RapidOCR] main.py:65: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-17 19:10:08,387 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 19:10:08,422 [RapidOCR] download_file.py:60: File exists and is valid: /usr

OK  RapidOCR API (text_det / text_rec present)
OK  person-seg ONNX (11488 KB)
GpuStackReport(torch_installed=True, torch_cuda=True, torch_cuda_device='Tesla T4', torch_mps=False, onnxruntime_installed=True, ort_providers=('AzureExecutionProvider', 'CPUExecutionProvider'), rapidocr_version='3.8.4', resolved_torch_device='cuda', resolved_ocr_backend='cpu')
GPU install hints:
OCR GPU (NVIDIA): pip install -e ".[ocr-gpu]"


/content/drive/MyDrive/Delete/SportsBiz/src/accelerator/install.py:197: RuntimeWarning: torch CUDA is available but CUDAExecutionProvider is not; OCR will run on CPU. Install with: pip uninstall -y onnxruntime onnxruntime-gpu && pip install onnxruntime-gpu
  resolved_ocr_backend=resolve_ocr_backend(accelerator),


## Configuration

`PipelineConfig` (`src/broadcast_pipeline/config.py`) controls every stage.

| Field | Default | Stage(s) | Description |
|-------|---------|----------|-------------|
| `video_path` | — | meta, extract | Input `.mp4` |
| `output_dir` | `data/pipeline` | all | Artifact root |
| `reference_csv` | `{output_dir}/approved_text_reference.csv` | reference, associate | Approved text list |
| `detector_threshold` | `27.0` | extract | PySceneDetect cut sensitivity (lower = more scenes) |
| `camera_samples_per_scene` | `5` | extract, cameras | Frames sampled per scene for clustering |
| `ocr_samples_per_sec` | `2.0` | extract | OCR frame sampling rate |
| `ensemble_method` | `"ensemble"` | cameras | `"hsv"`, `"embedding"`, or `"ensemble"` |
| `fast_cameras` | `False` | cameras | Force HSV-only when `True` |
| `appearance_enabled` | `True` | appearance | Run YOLO11n-seg person segmentation + clothing colors |
| `appearance_model_path` | `None` | appearance | ONNX path (default `{ROOT}/models/person_seg/yolo11n-seg.onnx`) |
| `appearance_imgsz` | `640` | appearance | YOLO input size |
| `appearance_min_confidence` | `0.5` | appearance | Detection confidence threshold |
| `appearance_color_tolerance` | `18.0` | appearance | LAB color distance for signature matching |
| `appearance_count_slack` | `1` | appearance | Allowed person-count mismatch in compatibility |
| `appearance_min_sequence_match` | `2` | appearance | Min subsequence length for close-up compatibility |
| `camera_stratify_by_appearance` | `True` | cameras | Stratify close-up clustering by appearance signature |
| `camera_appearance_reconcile` | `True` | cameras | Post-vote split incompatible close-up camera assignments |
| `appearance_feature_weight` | `0.15` | cameras | Soft appearance feature weight for `full_court` ensemble |
| `ocr_scale` | `1.5` | ocr | Upscale factor before OCR (lower = less RAM) |
| `ocr_preprocess` | `True` | ocr | CLAHE + preprocessing |
| `enable_vlm` | `False` | ocr | Escalate unread text to OpenAI VLM |
| `unk_token` | `"UNK"` | ocr, associate | Placeholder when text unreadable |
| `association_min_score` | `0.6` | associate | Min LCS score for partial match |
| `association_min_match_chars` | `3` | associate | Min LCS length |
| `enrich_enabled` | `True` | enrich | Temporal stabilization on/off |
| `enrich_region_iou` | `0.3` | enrich | IoU threshold for region tracking |
| `readability_size_multiplier` | `1.25` | enrich, aggregate | Readability area threshold |
| `default_new_text_approved` | `True` | reference | Auto-approve discovered tokens |
| `from_step` | `"all"` | — | Start stage for `run_pipeline` |
| `to_step` | `None` | — | End stage (inclusive); `None` runs through `aggregate` |
| `resume` | `False` | all | Skip stages whose outputs already exist |
| `accelerator` | `"auto"` | appearance, cameras, ocr | `"auto"`, `"cuda"`, `"mps"`, or `"cpu"` |

`video_meta.json` stores the **absolute** video path from probe time. Harmless on Colab unless you move the video file.


In [11]:
from broadcast_pipeline.config import PipelineConfig, STAGE_ORDER
from broadcast_pipeline.orchestrator import run_pipeline
from broadcast_pipeline.preflight import preflight

config = PipelineConfig(
    video_path=VIDEO_PATH,
    output_dir=OUTPUT_DIR,
    resume=True,
    from_step="all",
    **PROFILES[RUNTIME_PROFILE],
)
config.output_dir.mkdir(parents=True, exist_ok=True)

# Fail fast on missing dependencies
preflight(config, STAGE_ORDER)
print("Preflight OK — all dependencies available.")
print(f"Stages: {' → '.join(STAGE_ORDER)}")


/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(

https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(

http

Preflight OK — all dependencies available.
Stages: meta → extract → filter → appearance → cameras → ocr → reference → enrich → associate → aggregate


## Stage-by-stage deep dive

Each subsection explains one pipeline stage before you run it.

### Stage 1: `meta` — Video metadata

**Module:** `src/broadcast_pipeline/video_meta.py`

**Purpose:** Probe the input video for FPS, frame count, duration, and resolution. Downstream sampling (`ocr_samples_per_sec`, scene duration) depends on these values.

**Algorithm:** OpenCV `VideoCapture` for dimensions; PySceneDetect `open_video` for authoritative FPS/frame count when available.

**Output:** `video_meta.json`

**Colab notes:** Lightweight, seconds to run. No GPU required.


### Stage 2: `extract` — Scene detection & frame sampling

**Module:** `src/broadcast_pipeline/scene_extractor.py`

**Purpose:** Detect scene cuts, sample frames per scene for camera clustering and OCR, write JPEGs.

**Algorithm:**
- PySceneDetect `ContentDetector(threshold=detector_threshold)` finds cuts.
- Per scene, `camera_sample_frames` picks `camera_samples_per_scene` evenly spaced frames.
- `ocr_sample_frames` picks frames every `fps / ocr_samples_per_sec` frames.
- A single video scan writes each unique frame once (a frame can have both `camera` and `ocr` roles).

**Outputs:** `scenes.json`, `frame_index.csv`, `frames/scene_{id}_frame_{num}.jpg`

**Colab notes:** Full sequential video scan. Memory scales with resolution, not duration. Lower `ocr_samples_per_sec` to cut disk and downstream OCR cost.


### Stage 3: `filter` — Scene type classification

**Module:** `src/broadcast_pipeline/scene_filter.py` → `src/camera_assignemnt/scene_classifier/classifier.py`

**Purpose:** Classify each scene as `full_court` or `closeup` via majority vote over camera-sample frames.

**Algorithm:** HSV court-surface mask ratio; optional MLP at `models/scene_mlp.joblib` when present. Falls back to court-ratio heuristic without the model.

**Output:** `scene_types.csv`

**Colab notes:** Fast, CPU-only. Feeds the `appearance` stage (close-up vs full-court) and optional `camera_stratify_by_scene_type`.


### Stage 4: `appearance` — Person segmentation & clothing colors

**Module:** `src/broadcast_pipeline/appearance_filter.py` → `src/person_appearance/`

**Purpose:** Detect people in camera-sample frames, extract torso clothing colors, and build per-scene appearance signatures. Downstream camera clustering uses these signatures to keep incompatible close-up shots from sharing a `camera_id`.

**Algorithm:**
- YOLO11n-seg ONNX (`models/person_seg/yolo11n-seg.onnx`) segments persons via `onnxruntime`.
- Torso ROI colors are quantized in LAB space (gray-world skipped on high-saturation crops).
- Scene signature = left-to-right color sequence with tolerant subsequence matching (not strict slot-wise).
- Close-ups: hard constraints — stratified clustering pools + merge blocking + post-vote reconciliation.
- Full-court: soft boost — appearance features appended to the embedding ensemble vote.

**Outputs:** `frame_appearance.csv`, `scene_appearance.csv`

**Colab notes:** Moderate cost on camera-sample frames only. Export the ONNX once in the setup cell (`scripts/download_person_seg_model.py`). Set `appearance_enabled=False` to skip if you only need coarse camera IDs.

### Stage 5: `cameras` — Camera identity clustering

**Module:** `src/broadcast_pipeline/camera_assignment.py` → `src/camera_assignemnt/embedding_cluster/pipeline.py`

**Purpose:** Assign a `camera_id` to each sampled frame and scene via unsupervised clustering.

**Algorithm:**
- **Free profile (`hsv`):** Multi-region HSV histogram features + DBSCAN/HDBSCAN clustering.
- **Pro profile (`ensemble`):** HSV + ResNet50 + DINOv2 vote ensemble; models loaded sequentially with `gc.collect()` between members.
- **Appearance constraints:** Close-up scenes are stratified by `scene_appearance.csv` signatures; incompatible merges are blocked and post-vote reconciliation can split bad assignments. Full-court scenes get a soft appearance feature boost (`appearance_feature_weight`).
- Scene-level ID = majority vote over camera-sample frames.

**Outputs:** `scene_assignments.csv`, `frame_assignments.csv`

**Colab notes:** Heaviest GPU stage in Pro profile. First run downloads torchvision weights. Requires `scene_appearance.csv` when `appearance_enabled=True` — run Group 2 as written (`filter` → `cameras` includes `appearance`).


### Stage 6: `ocr` — On-screen text extraction

**Module:** `src/broadcast_pipeline/ocr_runner.py` → `src/scene_ocr/`

**Purpose:** Run RapidOCR on OCR-sample frames, assess readability, optionally escalate to VLM.

**Algorithm:**
1. RapidOCR (ONNX) extracts text lines → word tokens.
2. Readability checks overlay regions of interest:

| Region | Normalized coords | Typical content |
|--------|-------------------|-----------------|
| `score_bug` | (0, 0.82) – (0.45, 1.0) | Score / clock |
| `logo` | (0.85, 0) – (1.0, 0.12) | Network logo |
| `sponsor_band` | (0, 0.08) – (0.78, 0.62) | Sponsor graphics |

3. Unreadable regions → `UNK` token (default) or OpenAI VLM if `enable_vlm=True`.
4. Results appended incrementally to `frame_ocr.csv`.

**Output:** `frame_ocr.csv`

**Colab notes:**
- `gc.collect()` after every frame.
- **Frame-level resume:** `run_segment_ocr()` skips frames already in CSV via `ocr_done_keys()` — survives mid-stage interrupts even when stage-level `resume` behaves differently.
- Lower `ocr_scale` (1.0–1.5) to reduce RAM.


### Stage 7: `reference` — Text reference discovery

**Module:** `src/broadcast_pipeline/text_reference.py`

**Purpose:** Discover plausible OCR tokens and merge into a user-editable reference CSV with approval flags.

**Algorithm:** Scan `words_json` for plausible complete tokens; merge with existing reference; new entries get `approved` from `default_new_text_approved` (default `True`).

**Output:** `approved_text_reference.csv` (columns: `complete_text`, `approved`, `first_seen_scene_id`, `first_seen_frame`, `discovery_count`)

**Colab notes:** **Human-in-the-loop checkpoint.** Review the CSV before associate if you want to reject spurious tokens (`approved=false`).


### Stage 8: `enrich` — Temporal OCR stabilization

**Module:** `src/broadcast_pipeline/text_enrich.py`

**Purpose:** Track text regions within camera runs per scene using IoU, apply mode filtering, label readability (`good`/`partial`), carry missing detections forward.

**Output:** `frame_ocr_enriched.csv`

**Colab notes:** Pure pandas/numpy. Set `enrich_enabled=False` to pass through raw OCR. Downstream stages prefer enriched CSV when present.


### Stage 9: `associate` — Map OCR to reference text

**Module:** `src/broadcast_pipeline/text_associate.py`

**Purpose:** Match OCR tokens to approved reference entries.

**Algorithm:**
- Exact case-insensitive match → `text_kind=complete`
- Otherwise longest-common-subsequence partial match above `association_min_score`
- Drop: UNK tokens, unapproved reference, no match

**Outputs:** `frame_text_associated.csv`, `dropped_text.csv`

**Colab notes:** CPU-only string matching. Inspect `dropped_text.csv` to understand rejections.


### Stage 10: `aggregate` — Per-camera text timelines

**Module:** `src/broadcast_pipeline/aggregator.py`

**Purpose:** Build per-camera text timelines with duration, merged frame ranges, readability stats.

**Outputs:** `aggregated_complete.csv`, `aggregated_partial.csv`, `pipeline_summary.json`

**Colab notes:** Lightweight. Primary input for visualization.


## Chunked execution

`run_pipeline()` respects `from_step` and `to_step` on `PipelineConfig`. Setting only `from_step="meta"` (with `to_step=None`) runs through `aggregate` in one call.

This notebook defines `run_stages(config, from_step, until_step)` as a thin wrapper that sets both fields, enabling Colab-friendly chunks.

### Resume semantics

| Mechanism | Granularity | When |
|-----------|-------------|------|
| `config.resume=True` | Stage | Skips stage if output file exists |
| OCR `ocr_done_keys()` | Frame | Skips frames already in `frame_ocr.csv` |
| Human-loop re-run | Stage | After editing reference — use `resume=False` or delete downstream CSVs |

> **Trap:** After editing `approved_text_reference.csv`, `resume=True` **skips** associate/aggregate if `frame_text_associated.csv` exists. See [Human-in-the-loop](#human-in-the-loop) below.


In [12]:
from broadcast_pipeline.artifacts import resolve_stage_range
from broadcast_pipeline.config import PipelineConfig
from broadcast_pipeline.orchestrator import run_pipeline
from broadcast_pipeline.types import PipelineSummary


def run_stages(
    config: PipelineConfig,
    from_step: str,
    until_step: str,
) -> PipelineSummary:
    """Run pipeline stages from *from_step* through *until_step* (inclusive)."""
    slice_stages = resolve_stage_range(from_step=from_step, to_step=until_step)
    print(f"run_stages: {' → '.join(slice_stages)}")
    saved_from = config.from_step
    saved_to = config.to_step
    try:
        config.from_step = from_step
        config.to_step = until_step
        return run_pipeline(config)
    finally:
        config.from_step = saved_from
        config.to_step = saved_to


### Group 1: `meta` → `extract`

In [ ]:
summary_g1 = run_stages(config, from_step="meta", until_step="extract")
print("Artifacts:", {k: str(v) for k, v in summary_g1.artifacts.items()})
print(f"Scenes: {summary_g1.n_scenes}")


run_stages: meta → extract
[19:10:09] preflight: starting — 10 stage(s): meta→aggregate
[19:10:09] preflight: done
[19:10:09] meta: starting — Untitled_sample.mp4
[19:10:12] meta: done — 333.5s, 59.94 fps, 19992 frames, 1920x1080
[19:10:12] extract: starting — Untitled_sample.mp4
[19:10:12]   Detecting scene cuts (PySceneDetect progress below)...


  Detected: 0 | Progress:   0%|          | 0/19992 [00:00<?, ?frames/s]INFO:pyscenedetect:Detecting scenes...
  Detected: 38 | Progress: 100%|█████████▉| 19991/19992 [02:26<00:00, 136.06frames/s]

[19:12:39]   Detected 39 scene(s)
[19:12:39]   Extracting 834 unique frame(s) from video


[19:12:56]   Video scan: 2000/19992 (10%)
[19:13:12]   Video scan: 3999/19992 (20%)
[19:13:28]   Video scan: 5998/19992 (30%)
[19:13:46]   Video scan: 7997/19992 (40%)
[19:14:02]   Video scan: 9996/19992 (50%)
[19:14:17]   Video scan: 11996/19992 (60%)
[19:14:34]   Video scan: 13995/19992 (70%)
[19:14:50]   Video scan: 15994/19992 (80%)
[19:15:06]   Video scan: 17993/19992 (90%)
[19:15:21] extract: done — 39 scenes, 877 sampled frames
[19:15:22] filter: starting — 39 scenes
[19:15:22]   Scene classification: 10/195 (5%)
[19:15:23]   Scene classification: 20/195 (10%)
[19:15:24]   Scene classification: 30/195 (15%)
[19:15:24]   Scene classification: 39/195 (20%)
[19:15:25]   Scene classification: 49/195 (25%)
[19:15:26]   Scene classification: 59/195 (30%)
[19:15:26]   Scene classification: 69/195 (35%)
[19:15:27]   Scene classification: 78/195 (40%)
[19:15:27]   Scene classification: 88/195 (45%)
[19:15:28]   Scene classification: 98/195 (50%)
[19:15:28]   Scene classification: 108/195

  warnings.warn("xFormers is not available (SwiGLU)")

  warnings.warn("xFormers is not available (Attention)")

  warnings.warn("xFormers is not available (Block)")



...

  Ensemble member: HSV
  Ensemble member: resnet50
  Ensemble member: dinov2_vits14


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


[19:17:23] cameras: done — 19 camera(s) across 39 scenes
[19:17:23] ocr: starting — 682 frames
[19:17:23]   Accelerator: torch=cuda, OCR=CPU
[19:17:23]   Hint: OCR fell back to CPU while torch uses CUDA — install onnxruntime-gpu (pip install -e '.[ocr-gpu]')
[19:17:23]   OCR is running on CPU (install onnxruntime-gpu for CUDA acceleration)


[INFO] 2026-06-17 19:17:23,633 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 19:17:23,650 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-17 19:17:23,651 [RapidOCR] main.py:65: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-17 19:17:23,762 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 19:17:23,766 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-17 19:17:23,767 [RapidOCR] main.py:65: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-17 19:17:23,823 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 19:17:23,858 [RapidOCR] download_file.py:60: File exists and is valid: /usr

[19:19:50]   OCR: 35/682 (5%)
[19:22:14]   OCR: 69/682 (10%)


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

meta = json.loads(config.artifact("video_meta").read_text())
print(f"Duration: {meta['duration_sec']:.1f}s, FPS: {meta['fps']:.2f}, "
      f"Frames: {meta['frame_count']}, Size: {meta['width']}x{meta['height']}")

scenes = json.loads(config.artifact("scenes").read_text())
print(f"Detected {len(scenes)} scene(s)")

frame_index = pd.read_csv(config.artifact("frame_index"))
print(frame_index.groupby("sample_role").size())

sample = frame_index.iloc[0]
if Path(sample["frame_path"]).is_file():
    display(Image(filename=str(sample["frame_path"]), width=400))


### Group 2: `filter` → `appearance` → `cameras`

Runs scene-type classification, person appearance extraction, then camera clustering. `run_stages(..., until_step="cameras")` includes the `appearance` stage automatically.

In [ ]:
summary_g2 = run_stages(config, from_step="filter", until_step="cameras")
print(f"Cameras: {summary_g2.n_cameras}")


In [ ]:
scene_types = pd.read_csv(config.artifact("scene_types"))
scene_appearance = pd.read_csv(config.artifact("scene_appearance"))
scene_assignments = pd.read_csv(config.artifact("scene_assignments"))
print("Scene types:\n", scene_types.head())
print("\nAppearance signatures (close-ups):\n",
      scene_appearance[scene_appearance["scene_type"] == "closeup"].head())
print(f"\nCamera IDs: {sorted(scene_assignments['camera_id'].unique())}")


### Group 3: `ocr`

Safe to re-run if interrupted — frame-level resume skips completed frames.

Requires `rapidocr>=3.8` (installed in setup cell).

In [ ]:
# Clear cached OCR engine after pip install / version changes
import src.scene_ocr.extractor as _ocr_ext

_ocr_ext._get_engine.cache_clear()

summary_g3 = run_stages(config, from_step="ocr", until_step="ocr")
print(f"OCR frames: {summary_g3.n_frames}")


In [ ]:
frame_ocr = pd.read_csv(config.artifact("frame_ocr"))
print(f"Rows: {len(frame_ocr)}, columns: {list(frame_ocr.columns)}")
if len(frame_ocr):
    print(frame_ocr[["scene_id", "frame_number", "camera_id", "verdict", "used_unk"]].head())


### Group 4: `reference` — pause for review

`default_new_text_approved=True` means discovered tokens are auto-approved. Set `approved=false` for any spurious entries before running Group 5.

Download/edit `approved_text_reference.csv` from Drive if needed.


In [ ]:
summary_g4 = run_stages(config, from_step="reference", until_step="reference")
ref = pd.read_csv(config.reference_csv)
print(f"Reference entries: {len(ref)} ({ref['approved'].sum()} approved)")
ref.head(10)


### Group 5: `enrich` → `associate` → `aggregate`

In [ ]:
summary_g5 = run_stages(config, from_step="enrich", until_step="aggregate")
print("Pipeline complete.")
for name, path in summary_g5.artifacts.items():
    print(f"  {name}: {path}")


## Human-in-the-loop

After editing `approved_text_reference.csv`:

1. Set `config.from_step = "associate"`
2. Set `config.resume = False` (**required** — otherwise associate is skipped)
3. Run `run_pipeline(config)` or `run_stages(config, "associate", "aggregate")`

Alternatively delete: `frame_text_associated.csv`, `dropped_text.csv`, `aggregated_complete.csv`, `aggregated_partial.csv`, `pipeline_summary.json`


In [ ]:
# Uncomment after editing approved_text_reference.csv:
# config.from_step = "associate"
# config.resume = False
# summary_rerun = run_stages(config, from_step="associate", until_step="aggregate")


## Full end-to-end run

Alternative to chunked execution — one cell runs everything (best on Pro with short clips).


In [ ]:
# config.from_step = "all"
# config.to_step = None
# config.resume = True
# summary = run_pipeline(config)
# print(f"Scenes={summary.n_scenes}, Cameras={summary.n_cameras}, OCR frames={summary.n_frames}")


## Results exploration

### Aggregated timeline columns

| Column | Meaning |
|--------|---------|
| `camera_id` | Clustered camera identifier |
| `text` | Observed text (complete or partial) |
| `text_kind` | `complete` or `partial` |
| `mapped_complete_text` | Reference text for partial matches |
| `total_duration_sec` | Weighted time on screen |
| `frame_ranges` | Semicolon-separated frame ranges |
| `n_frames_present` | Frames with this text |
| `n_frames_good` / `n_frames_partial` | Readability breakdown |
| `n_frames_enriched` | Frames where enrich was applied |
| `dominant_readability` | Majority readability label |


In [ ]:
import json
import pandas as pd
from IPython.display import display


def _safe_read_csv(path):
    if not path.is_file() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


complete = _safe_read_csv(config.artifact("aggregated_complete"))
partial = _safe_read_csv(config.artifact("aggregated_partial"))
dropped = _safe_read_csv(config.artifact("dropped_text"))
summary_json = json.loads(config.artifact("pipeline_summary").read_text())

print("=== Pipeline summary ===")
for k, v in summary_json.items():
    print(f"  {k}: {v}")

print(f"\nComplete rows: {len(complete)}, Partial rows: {len(partial)}, Dropped: {len(dropped)}")
if len(complete):
    display(complete.head(10))
if len(partial):
    print("\nPartial text (sample):")
    display(partial.head(5))
if len(dropped):
    print("\nDropped text (sample):")
    display(dropped.head(5))


In [ ]:
# Timeline bar chart per camera (matplotlib — reliable on Colab)
import matplotlib.pyplot as plt

if len(complete):
    fig, ax = plt.subplots(figsize=(12, max(3, complete["camera_id"].nunique() * 0.5)))
    cameras = sorted(complete["camera_id"].unique())
    y_map = {c: i for i, c in enumerate(cameras)}
    for _, row in complete.iterrows():
        y = y_map[row["camera_id"]]
        ax.barh(y, row["total_duration_sec"], left=0, height=0.6, alpha=0.7)
        ax.text(0.1, y, str(row["text"])[:30], va="center", fontsize=8)
    ax.set_yticks(range(len(cameras)))
    ax.set_yticklabels(cameras)
    ax.set_xlabel("Duration (sec)")
    ax.set_title("Complete text timeline by camera")
    plt.tight_layout()
    plt.show()
else:
  print("No complete rows to chart.")


## Optional timeline web UI

Requires `pip install fastapi uvicorn python-multipart` and `static/timeline_viz/` in your zip.

**CORS limitation:** `create_app()` only allows `localhost` origins. On Colab, use `serve_kernel_port_as_window` (iframe) rather than opening the raw proxy URL. If API calls fail, Part 7 matplotlib charts are the reliable fallback.


In [ ]:
# %pip install -q fastapi uvicorn python-multipart

# import threading
# import uvicorn
# from broadcast_pipeline.viz.server import create_app

# app = create_app(OUTPUT_DIR.resolve(), (ROOT / "static" / "timeline_viz").resolve())

# def _serve():
#     uvicorn.run(app, host="0.0.0.0", port=8765, log_level="warning")

# threading.Thread(target=_serve, daemon=True).start()

# try:
#     from google.colab import output
#     output.serve_kernel_port_as_window(8765, path="/", height=800)
# except ImportError:
#     print("Open http://127.0.0.1:8765 in a local browser")


## Standalone OCR debug

For single-frame OCR outside the full pipeline, use `scripts/run_ocr.py`:

```bash
python scripts/run_ocr.py --image path/to/frame.jpg --assess --json
```

Install: `pip install "rapidocr>=3.8" onnxruntime` (+ `openai` for `--openai-vlm`).


## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `ImportError: src.camera_assignemnt` | Missing ROOT on sys.path | Add repo root: `sys.path.insert(0, str(ROOT))` |
| CUDA OOM on cameras | Ensemble on small GPU | Use free profile or lower `camera_samples_per_scene` |
| OCR slow / RAM spikes | High `ocr_scale` or `ocr_samples_per_sec` | Lower both in profile |
| Empty associated text | Unapproved reference entries | Set `approved=true` in reference CSV |
| Session disconnect | Long video | Drive output + `resume=True` + chunked `run_stages` |
| Missing rapidocr | OCR extra not installed | `%pip install "rapidocr>=3.8" onnxruntime` |
| RapidOCR AttributeError on OCR | Old extractor or rapidocr <3.8 | Copy updated `src/scene_ocr/extractor.py`; `pip install -U rapidocr>=3.8` |
| ROOT exists but import fails | Empty/wrong folder | Unzip cell checks `src/broadcast_pipeline`; set `FORCE_REEXTRACT=True` |
| torch required despite `fast_cameras` | preflight checks `ensemble_method` | Set `ensemble_method="hsv"` in free profile |
| Associate unchanged after CSV edit | `resume=True` skips existing outputs | `resume=False` or delete downstream CSVs |
| Viz UI loads but API fails | Colab CORS | Use `serve_kernel_port_as_window`; see Part 8 |
| First cameras run slow | Model download | Needs network; one-time per session |
| RapidOCR first run slow | ONNX download | Needs network; cached after first use |
| OCR still on CPU with GPU runtime | CPU `onnxruntime` installed | `%pip uninstall -y onnxruntime onnxruntime-gpu && %pip install onnxruntime-gpu` |
| `CUDAExecutionProvider` missing | onnxruntime-gpu not installed | Pro setup cell installs gpu wheel; verify with `check_gpu_stack()` |
| torch not using GPU | CPU torch wheel | Pro: `%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121` |
| `appearance` fails / missing ONNX | YOLO11n-seg not exported | Re-run setup cell or `python scripts/download_person_seg_model.py` |
| Close-up scenes share `camera_id` | Appearance disabled or stale artifacts | Keep `appearance_enabled=True`; delete `scene_appearance.csv` and re-run Group 2 |
| `scene_types` seems unused | Misconception | Feeds appearance constraints and optional `camera_stratify_by_scene_type` |


## Out of scope

The following exist in the repo but are **not** part of `run_pipeline()`:

- `src/camera_split_segment.py` — legacy frame-extraction helper
- `homography_projector.py`, `scripts/eval_homography.py` — court homography subsystem
- `scripts/train_scene_classifier.py`, `scripts/run_camera_clustering.py` — offline model training
